# EDA: Financial Precarity Simulation Results
## Phân tích bất bình đẳng và thống kê mô phỏng 10,000 agents

Notebook này load dữ liệu kết quả từ mô phỏng `VectorizedFinancialEnv` và thực hiện:
1. Hệ số Gini theo thời gian
2. Thống kê cuối kỳ (tháng 60)
3. Phân tích bất bình đẳng giàu-nghèo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['legend.fontsize'] = 14
matplotlib.rcParams['axes.labelsize'] = 16
matplotlib.rcParams['axes.titlesize'] = 18
matplotlib.rcParams['xtick.labelsize'] = 12
matplotlib.rcParams['ytick.labelsize'] = 12
sns.set_theme(style='whitegrid')

In [ ]:
# Chạy mô phỏng (hoặc load lại nếu đã có sẵn)
from vectorized_env import VectorizedFinancialEnv

N_AGENTS = 10_000
T = 60

env = VectorizedFinancialEnv(n_agents=N_AGENTS, seed=42, r=1.02)
history = env.run_episode(T=T, return_history=True)
print(f"Simulation complete: {N_AGENTS} agents x {T} months")
print(f"Bankruptcy rate: {env.bankruptcy_rate:.2%}")

## 1. Hệ số Gini theo thời gian

Hệ số Gini đo lường bất bình đẳng: 0 = hoàn toàn bình đẳng, 1 = hoàn toàn bất bình đẳng.

In [ ]:
def gini(x):
    x = np.sort(x)
    n = len(x)
    cumsum = np.cumsum(x)
    if cumsum[-1] <= 0:
        return 0.0
    return (2 * np.sum(np.arange(1, n + 1) * x) - (n + 1) * cumsum[-1]) / (n * cumsum[-1])

# Tính Gini tại mỗi thời điểm
gini_over_time = []
for t in range(history.shape[0]):
    gini_over_time.append(gini(history[t]))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(gini_over_time)), gini_over_time, 'purple', linewidth=2)
ax.axhline(gini_over_time[0], color='gray', linestyle='--', alpha=0.5, label=f'Initial Gini = {gini_over_time[0]:.4f}')
ax.axhline(gini_over_time[-1], color='red', linestyle='-.', alpha=0.7, label=f'Final Gini = {gini_over_time[-1]:.4f}')
ax.set_xlabel('Time (months)')
ax.set_ylabel('Gini Coefficient')
ax.set_title('Gini Coefficient Over Time')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Thống kê kết quả cuối cùng (tháng 60)

In [ ]:
final_assets = history[-1]
initial_assets = history[0]

print("=" * 60)
print("THỐNG KÊ KẾT QUẢ CUỐI CÙNG (THÁNG 60)")
print("=" * 60)
print(f"{'Statistic':<30} {'Value':>12}")
print("-" * 42)
print(f"{'Total agents':<30} {N_AGENTS:>12}")
print(f"{'Bankrupt':<30} {int(env.bankrupt.sum()):>12}")
print(f"{'Bankruptcy rate':<30} {env.bankruptcy_rate:>12.2%}")
print(f"{'Gini coefficient':<30} {gini_over_time[-1]:>12.4f}")
print(f"{'Mean assets (all)':<30} {final_assets.mean():>12.4f}")
print(f"{'Median assets (all)':<30} {np.median(final_assets):>12.4f}")
print(f"{'Std assets (all)':<30} {final_assets.std():>12.4f}")
print(f"{'Max assets':<30} {final_assets.max():>12.4f}")
print(f"{'Min assets (non-zero)':<30} {final_assets[final_assets > 0].min():>12.4f}")
print(f"{'Initial Gini':<30} {gini_over_time[0]:>12.4f}")
print(f"{'Gini change':<30} {gini_over_time[-1] - gini_over_time[0]:>+12.4f}")

## 3. Phân phối tài sản: Đầu kỳ vs Cuối kỳ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Biểu đồ 1: Histogram tài sản đầu kỳ vs cuối kỳ
ax1 = axes[0]
ax1.hist(initial_assets, bins=80, alpha=0.5, color='green', label=f'Initial (t=0)')
ax1.hist(final_assets, bins=80, alpha=0.5, color='red', label=f'Final (t={T})')
ax1.set_xlabel('Assets')
ax1.set_ylabel('Count')
ax1.set_title('Asset Distribution: Initial vs Final')
ax1.legend()
ax1.grid(alpha=0.3)

# Biểu đồ 2: Log-scale
ax2 = axes[1]
ax2.hist(initial_assets[initial_assets > 0], bins=80, alpha=0.5, color='green', label=f'Initial (t=0)')
ax2.hist(final_assets[final_assets > 0], bins=80, alpha=0.5, color='red', label=f'Final (t={T})')
ax2.set_xscale('log')
ax2.set_xlabel('Assets (log scale)')
ax2.set_ylabel('Count')
ax2.set_title('Asset Distribution (log scale)')
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 4. Phân tích bất bình đẳng: Phân vị tài sản

In [ ]:
percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99]
init_pct = np.percentile(initial_assets, percentiles)
final_pct = np.percentile(final_assets, percentiles)

print("=" * 64)
print(f"{'Percentile':<12} {'Initial':>12} {'Final':>12} {'Change':>12} {'% Change':>12}")
print("-" * 64)
for i, p in enumerate(percentiles):
    change = final_pct[i] - init_pct[i]
    pct_ch = (change / init_pct[i]) * 100 if init_pct[i] != 0 else float('inf')
    print(f"{'P' + str(p):<12} {init_pct[i]:>12.4f} {final_pct[i]:>12.4f} {change:>+12.4f} {pct_ch:>11.2f}%")
print("=" * 64)

## 5. Lorenz Curve (Cuối kỳ)

Đường cong Lorenz minh họa bất bình đẳng: đường càng xa đường 45 độ, bất bình đẳng càng cao.

In [ ]:
def lorenz_curve(x):
    x = np.sort(x)
    cum = np.cumsum(x)
    cum = cum / cum[-1] if cum[-1] > 0 else cum
    return np.insert(cum, 0, 0)

n = len(final_assets)
pop_pct = np.linspace(0, 1, n + 1)

lorenz_init = lorenz_curve(initial_assets)
lorenz_final = lorenz_curve(final_assets)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(pop_pct, lorenz_init, 'g-', linewidth=2, label=f'Initial (Gini={gini_over_time[0]:.4f})')
ax.plot(pop_pct, lorenz_final, 'r-', linewidth=2, label=f'Final (Gini={gini_over_time[-1]:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Equality')
ax.fill_between(pop_pct, lorenz_final, pop_pct, alpha=0.1, color='red')
ax.set_xlabel('Cumulative Population Share')
ax.set_ylabel('Cumulative Asset Share')
ax.set_title('Lorenz Curve: Initial vs Final')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_aspect('equal')
fig.tight_layout()
plt.show()

## 6. Top Wealth Share

Phần trăm tài sản nắm giữ bởi nhóm giàu nhất.

In [ ]:
for top_pct in [1, 5, 10, 20]:
    n_top = int(N_AGENTS * top_pct / 100)
    sorted_init = np.sort(initial_assets)[::-1]
    sorted_final = np.sort(final_assets)[::-1]
    share_init = sorted_init[:n_top].sum() / initial_assets.sum() * 100
    share_final = sorted_final[:n_top].sum() / final_assets.sum() * 100
    print(f"Top {top_pct:>2}% holds: Initial={share_init:.2f}%, Final={share_final:.2f}%")

# Top/bottom 10% ratio
top10_init = np.sort(initial_assets)[-int(0.1 * N_AGENTS):].sum()
bot10_init = np.sort(initial_assets)[:int(0.1 * N_AGENTS)].sum()
top10_final = np.sort(final_assets)[-int(0.1 * N_AGENTS):].sum()
bot10_final = np.sort(final_assets)[:int(0.1 * N_AGENTS)].sum()

print(f"\nTop 10% / Bottom 10% ratio (Initial): {top10_init / bot10_init:.2f}")
print(f"Top 10% / Bottom 10% ratio (Final):   {top10_final / bot10_final:.2f}")

## 7. Tương quan Giữa Thu Nhập và Tài Sản Cuối Kỳ

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(env.incomes, final_assets, c=final_assets, cmap='viridis', 
                alpha=0.3, s=5, edgecolors='none')
ax.set_xlabel('Initial Income')
ax.set_ylabel('Final Assets')
ax.set_title('Initial Income vs Final Assets')
plt.colorbar(sc, ax=ax, label='Final Assets')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# Tính correlation
alive_mask = ~env.bankrupt
corr = np.corrcoef(env.incomes[alive_mask], final_assets[alive_mask])[0, 1]
print(f"Correlation (income vs final assets, survivors): {corr:.4f}")

## 8. Kết luận

Từ phân tích EDA trên, chúng ta có thể rút ra:
- **Bất bình đẳng gia tăng**: Hệ số Gini tăng từ {gini_over_time[0]:.4f} lên {gini_over_time[-1]:.4f} sau {T} tháng.
- **Tỷ lệ phá sản**: {env.bankruptcy_rate:.2%} agents bị phá sản.
- **Tập trung tài sản**: Nhóm giàu nhất nắm giữ phần lớn tài sản.
- **Tương quan thu nhập-tài sản**: Thu nhập ban đầu có tương quan {corr:.4f} với tài sản cuối kỳ.